#Social Listening Engine

##1. Setup and data loading

First things first — mount Google Drive and create project folders.
This keeps everything organized and saves our work between sessions.

In [3]:

from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = "/content/drive/MyDrive/social_listening_engine"
DIRS = {
    "data": f"{PROJECT_DIR}/data",
    "indexes": f"{PROJECT_DIR}/indexes",
    "reports": f"{PROJECT_DIR}/reports",
    "models": f"{PROJECT_DIR}/models",
    "feedback": f"{PROJECT_DIR}/feedback",
}

for k, path in DIRS.items():
    os.makedirs(path, exist_ok=True)

print("Project folders created:")
for k, path in DIRS.items():
    print(f" - {k}: {path}")


!pip -q install faiss-cpu

import pandas as pd

DATA_PATH = f"{PROJECT_DIR}/data/events_with_topics.parquet"

meta = pd.read_parquet(DATA_PATH)

print(meta.shape)
print(meta.columns.tolist())

import json

TOPICS_PATH = f"{PROJECT_DIR}/data/topics.json"

with open(TOPICS_PATH, "r") as f:
    topics_data = json.load(f)

topics_by_id = {t["topic_id"]: t for t in topics_data}

print("Loaded topics:", len(topics_by_id))



import faiss
print("faiss:", faiss.__version__)

import faiss
import numpy as np


from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print(tok("hello world"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folders created:
 - data: /content/drive/MyDrive/social_listening_engine/data
 - indexes: /content/drive/MyDrive/social_listening_engine/indexes
 - reports: /content/drive/MyDrive/social_listening_engine/reports
 - models: /content/drive/MyDrive/social_listening_engine/models
 - feedback: /content/drive/MyDrive/social_listening_engine/feedback
(45615, 8)
['event_id', 'timestamp', 'source', 'sentiment_label', 'intent_label_final', 'text_clean', 'row_id', 'topic_id']
Loaded topics: 20
faiss: 1.13.2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}


In [ ]:
# the check versions and import the right things
import sys
print("Python:", sys.version)

import numpy as np
import pandas as pd
import torch

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy: 2.0.2
pandas: 2.2.2
torch: 2.9.0+cpu


In [ ]:
# These may or may not exist yet — that's fine
try:
    import datasets
    print("datasets:", datasets.__version__)
except Exception as e:
    print("datasets not available")

try:
    import transformers
    print("transformers:", transformers.__version__)
except Exception as e:
    print("transformers not available")

try:
    import sentence_transformers
    print("sentence-transformers:", sentence_transformers.__version__)
except Exception as e:
    print("sentence-transformers not available")

try:
    import faiss
    print("faiss:", faiss.__version__)
except Exception as e:
    print("faiss not available")


datasets: 4.0.0
transformers: 5.0.0
sentence-transformers: 5.2.2
faiss: 1.13.2


In [ ]:
!pip -q install faiss-cpu

In [ ]:
import faiss
print("faiss:", faiss.__version__)

faiss: 1.13.2


In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print(tok("hello world"))


{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}


## Loading Dataset

We're using tweet_eval sentiment dataset — about 45k tweets with sentiment labels.
Good enough for demo, though in real life you'd stream live data.

In [ ]:
from datasets import load_dataset

N = 50_000
ds = load_dataset("tweet_eval", "sentiment", split=f"train[:{N}]")

print(ds)
print("Columns:", ds.column_names)
print("Row 0:", ds[0])


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 45615
})
Columns: ['text', 'label']
Row 0: {'text': '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', 'label': 2}


tweet_eval does not include timestamps.

So i've:

- generated synthetic timestamps

- confined them to a realistic window (last ~90 days)

- planned to flag them as synthetic in exports

This lets us:

- build trend logic

- test aggregation code

- design dashboards
without lying to ourselves or future reviewers.


In [ ]:
import pandas as pd
import numpy as np

df = ds.to_pandas()

# Synthetic timestamps for demo (explicitly marked)
end = pd.Timestamp.today().normalize()
start = end - pd.Timedelta(days=90)

df["timestamp"] = pd.to_datetime(
    np.random.randint(start.value // 10**9, end.value // 10**9, size=len(df)),
    unit="s"
)
df["timestamp_is_synthetic"] = True

# Standard event schema (minimum for now)
events = pd.DataFrame({
    "event_id": np.arange(len(df)).astype("int64"),
    "text": df["text"].astype(str),
    "timestamp": df["timestamp"],
    "timestamp_is_synthetic": df["timestamp_is_synthetic"],
    "source": "tweet_eval",
    "sentiment_label_raw": df["label"],  # still numeric for now
})

events.head(10)


,event_id,text,timestamp,timestamp_is_synthetic,source,sentiment_label_raw
0,0,"""QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin""",2026-01-07 16:32:37,True,tweet_eval,2
1,1,"""Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ""",2025-12-06 05:19:00,True,tweet_eval,1
2,2,Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.,2026-01-23 05:13:47,True,tweet_eval,1
3,3,Chase Headley's RBI double in the 8th inning off David Price snapped a Yankees streak of 33 consecutive scoreless innings against Blue Jays,2026-01-26 13:30:02,True,tweet_eval,1
4,4,"@user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017""",2026-02-07 18:13:44,True,tweet_eval,2
5,5,@user LIT MY MUM 'Kerry the louboutins I wonder how many Willam owns!!! Look Kerry Warner Wednesday!',2025-12-16 09:02:28,True,tweet_eval,2
6,6,"""\"""""""" SOUL TRAIN\"""""""" OCT 27 HALLOWEEN SPECIAL ft T.dot FINEST rocking the mic...CRAZY CACTUS NIGHT CLUB ..ADV ticket $10 wt out costume $15...""",2026-02-08 04:31:44,True,tweet_eval,2
7,7,So disappointed in wwe summerslam! I want to see john cena wins his 16th title,2025-12-28 02:28:05,True,tweet_eval,0
8,8,"""This is the last Sunday w/o football .....,NFL is back baby""",2026-01-01 03:06:24,True,tweet_eval,2
9,9,@user @user CENA & AJ sitting in a tree K-I-S-S-I-N-G 1st goes AJ's job then John's cred then goes Vicki with the GM position.,2026-01-14 17:23:34,True,tweet_eval,1


Mapping the sentiments

In [ ]:
label_map = {0:'neg', 1:'neutral', 2:'pos'}
events['sentiment_label'] = events['sentiment_label_raw'].map(label_map).fillna('unknown')

print('Sentiment Distribution:')
print(events['sentiment_label'].value_counts())

events[['text','sentiment_label_raw','sentiment_label']].sample(10, random_state=51)

Sentiment Distribution:
sentiment_label
neutral    20673
pos        17849
neg         7093
Name: count, dtype: int64


,text,sentiment_label_raw,sentiment_label
12568,Pit Bull fans celebrate 1st annual Awareness Day: Animal lovers of all kinds gathered at the Buddy Holly Recreation...,2,pos
38348,"@user loved the way the debate on Yakub's hanging was ended by you. """"We the people"""" today was thought provoking",2,pos
32843,i have to drive with Ms. Pope Saturday... kill me now.,0,neg
39839,The brilliant Drew Westen offers a great critique of the Obama presidency:,2,pos
4437,Thank God we now have August 9th as the day Frank Gifford died instead of it being the day Michael Brown died.,1,neutral
5852,@user Happy birthday!! Have a top day. And hope Everton batter Chelsea on Saturday for you haha! x,2,pos
20306,"And it say ""April: Snoop Dogg quits weed!"" Then it says ""May: Snoop Dogg back on weed!"" I feel like that's me about a lot of things lmao",1,neutral
40487,"Rumor is that Apple's big announcement tomorrow is a giant iPad. Please, God -- please let them call it the MaxiPad! #AppleEvent #Apple",2,pos
30485,My soup just blew up in the microwave and exploded everywhere. So far today has been a bigger bitch than Madonna is to hydrangeas.,0,neg
11708,I do love my iPhone 6plus but I am definitely not a fan of the fact that I just accidentally sat on it and it is now another shape...,0,neg


## 2. Text Processing

Tweets are messy — mentions, URLs, hashtags.
Just stripping those out so our models see clean text.
Nothing fancy here, keep it simple.

In [ ]:
import re


def clean_text_basic(s: str) -> str:
  s = s.strip()
  s = re.sub(r"\s+", " ", s)
  s = re.sub(r"http\S+|www\.\S+", "", s)
  s = re.sub(r"@\w+", "@user", s)
  s = s.replace("#", "")
  return s.strip()

events["text_clean"] = events['text'].map(clean_text_basic)

events[['text', 'text_clean']].sample(10, random_state=51)

,text,text_clean
12568,Pit Bull fans celebrate 1st annual Awareness Day: Animal lovers of all kinds gathered at the Buddy Holly Recreation...,Pit Bull fans celebrate 1st annual Awareness Day: Animal lovers of all kinds gathered at the Buddy Holly Recreation...
38348,"@user loved the way the debate on Yakub's hanging was ended by you. """"We the people"""" today was thought provoking","@user loved the way the debate on Yakub's hanging was ended by you. """"We the people"""" today was thought provoking"
32843,i have to drive with Ms. Pope Saturday... kill me now.,i have to drive with Ms. Pope Saturday... kill me now.
39839,The brilliant Drew Westen offers a great critique of the Obama presidency:,The brilliant Drew Westen offers a great critique of the Obama presidency:
4437,Thank God we now have August 9th as the day Frank Gifford died instead of it being the day Michael Brown died.,Thank God we now have August 9th as the day Frank Gifford died instead of it being the day Michael Brown died.
5852,@user Happy birthday!! Have a top day. And hope Everton batter Chelsea on Saturday for you haha! x,@user Happy birthday!! Have a top day. And hope Everton batter Chelsea on Saturday for you haha! x
20306,"And it say ""April: Snoop Dogg quits weed!"" Then it says ""May: Snoop Dogg back on weed!"" I feel like that's me about a lot of things lmao","And it say ""April: Snoop Dogg quits weed!"" Then it says ""May: Snoop Dogg back on weed!"" I feel like that's me about a lot of things lmao"
40487,"Rumor is that Apple's big announcement tomorrow is a giant iPad. Please, God -- please let them call it the MaxiPad! #AppleEvent #Apple","Rumor is that Apple's big announcement tomorrow is a giant iPad. Please, God -- please let them call it the MaxiPad! AppleEvent Apple"
30485,My soup just blew up in the microwave and exploded everywhere. So far today has been a bigger bitch than Madonna is to hydrangeas.,My soup just blew up in the microwave and exploded everywhere. So far today has been a bigger bitch than Madonna is to hydrangeas.
11708,I do love my iPhone 6plus but I am definitely not a fan of the fact that I just accidentally sat on it and it is now another shape...,I do love my iPhone 6plus but I am definitely not a fan of the fact that I just accidentally sat on it and it is now another shape...


###adding review fields (for future)


In [ ]:
events['needs_review'] = False
events['intent_label'] = None
events['intent_conf'] = np.nan
events['sentiment_conf'] = np.nan



In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/social_listening_engine"
DIRS = {
    "data": f"{PROJECT_DIR}/data",
    "feedback": f"{PROJECT_DIR}/feedback",
}
for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

out_path = f"{DIRS['data']}/events_clean_small.parquet"
events.to_parquet(out_path, index=False)

print("Saved:", out_path)

# Read-back sanity
reloaded = pd.read_parquet(out_path)
print("Reloaded shape:", reloaded.shape)
print("Reloaded columns:", list(reloaded.columns))
reloaded.head(3)


Saved: /content/drive/MyDrive/social_listening_engine/data/events_clean_small.parquet
Reloaded shape: (45615, 12)
Reloaded columns: ['event_id', 'text', 'timestamp', 'timestamp_is_synthetic', 'source', 'sentiment_label_raw', 'sentiment_label', 'text_clean', 'needs_review', 'intent_label', 'intent_conf', 'sentiment_conf']


,event_id,text,timestamp,timestamp_is_synthetic,source,sentiment_label_raw,sentiment_label,text_clean,needs_review,intent_label,intent_conf,sentiment_conf
0,0,"""QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin""",2026-01-07 16:32:37,True,tweet_eval,2,pos,"""QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. HappyBirthdayRemusLupin""",False,None,NaN,NaN
1,1,"""Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ""",2025-12-06 05:19:00,True,tweet_eval,1,neutral,"""Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis NHL SJ""",False,None,NaN,NaN
2,2,Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.,2026-01-23 05:13:47,True,tweet_eval,1,neutral,Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.,False,None,NaN,NaN


##3. Intent Classification

###- Rule Based approach

We want to know if someone's complaining, asking a question, or just chatting.
Starting with keyword rules — quick and surprisingly effective.

In [ ]:
import re

# making a copy to avoid accidental issues in future
events2 = reloaded.copy()

#backing up current
events2["intent_label_prev"] = events2["intent_label"]
events2["intent_conf_prev"]  = events2["intent_conf"]

INTENT_RULE_VERSION = "rules_v1"

KW = {
    "question": [
        r"\?$", r"\bhow\b", r"\bwhy\b", r"\bwhat\b", r"\bwhen\b", r"\bwhere\b", r"\bwho\b",
        r"\bcan you\b", r"\bcould you\b", r"\bis it\b", r"\bare you\b", r"\bany update\b"
    ],
    "request": [
        r"\bplease\b", r"\bplz\b", r"\bkindly\b",
        r"\bneed\b", r"\bwant\b", r"\brequire\b",
        r"\bhelp\b", r"\bsend\b", r"\bshare\b", r"\bprovide\b",
        r"\brefund\b", r"\breturn\b", r"\breplace\b", r"\bcancel\b", r"\bexchange\b",
        r"\bfix\b", r"\bresolve\b"
    ],
    "complaint": [
        r"\bworst\b", r"\bterrible\b", r"\bbad\b", r"\bawful\b", r"\bpathetic\b",
        r"\blate\b", r"\bdelay\b", r"\bdelayed\b",
        r"\bnot working\b", r"\bbroken\b", r"\bscam\b", r"\bfraud\b",
        r"\brude\b", r"\bdisappointed\b", r"\bangry\b"
    ],
    "praise": [
        r"\blove\b", r"\bamazing\b", r"\bawesome\b", r"\bgreat\b", r"\bfantastic\b",
        r"\bthank(s| you)\b", r"\bexcellent\b", r"\bbrilliant\b", r"\bperfect\b"
    ],
}

PRIORITY = ["question", "request", "complaint", "praise"]
PATTERNS = {k: [re.compile(p, re.IGNORECASE) for p in v] for k, v in KW.items()}

def rule_intent(text: str):
    t = text or ""
    for cat in PRIORITY:
        hits = [p.pattern for p in PATTERNS[cat] if p.search(t)]
        if hits:
            conf = min(0.55 + 0.15 * len(hits), 0.95)
            return cat, conf, hits[:3]
    return "other", 0.40, []

tmp = events2["text_clean"].map(rule_intent)

events2["intent_rule_version"] = INTENT_RULE_VERSION
events2["intent_label_rule"]   = tmp.map(lambda x: x[0])
events2["intent_conf_rule"]    = tmp.map(lambda x: x[1])
events2["intent_reason_rule"]  = tmp.map(lambda x: x[2])

# setting final intent fields from rules
events2["intent_label"] = events2["intent_label_rule"]
events2["intent_conf"]  = events2["intent_conf_rule"]

# review flags
events2["needs_review"] = (events2["intent_conf"] < 0.55) | (events2["intent_label"] == "other")

print("Intent distribution (rules):")
print(events2["intent_label"].value_counts())

other_rate = (events2["intent_label"] == "other").mean()
print("Other rate %:", round(other_rate * 100, 2))

events2[["text_clean","intent_label","intent_conf","intent_reason_rule"]].sample(10, random_state=2)


Intent distribution (rules):
intent_label
other        31989
question      8062
praise        2398
request       2328
complaint      838
Name: count, dtype: int64
Other rate %: 70.13


,text_clean,intent_label,intent_conf,intent_reason_rule
39339,What's tomorrow?that's right the day my boys Jack and Jack release Calibraska and we're all gonna die because we're not ready,question,0.7,[\bwhat\b]
198,"""Best day and weekend trips from Seoul, by season: by Elizabeth EunThere's just so much to see in Seoul, it may t...",other,0.4,[]
2265,@user I wrote a column about the Royals for a site that you may be interested in reading:,other,0.4,[]
37429,Thank you St David for $7 parking on a home game Saturday. aff2012,praise,0.7,[\bthank(s| you)\b]
38922,Prayers for Yakub at the Mahim dargah just ahead of burial this is the state of Indian Muslims..they support them..,other,0.4,[]
12599,"@user yes, it's true I've missed all my children but no Go Set a Watchman gets released Tuesday""",other,0.4,[]
22856,Haha I'm going to see Foo Fighters tomorrow Jesus Christ that's just hit me,other,0.4,[]
33704,Lloyd Robertson set to end 41-year run as national news anchor tonight - News957: @user @user,other,0.4,[]
25294,"@user @user However, I don't think today's SCOTUS and Fed Govt would interpret the 9th that way. 9th &amp;10th have seemingly disappeared",other,0.4,[]
1312,Boom! 1st post Bilbao night half marathon 9km. I just finished a 9.02 km run with Nike+ Running. nikeplus,other,0.4,[]


###Improving Rule based approach
I sampled 300 posts under the 'other' intent_label and used an LLM to suggest better keywords.


In [ ]:

import pandas as pd

other_sample = events2[events2["intent_label"] == "other"][["text_clean"]].sample(300, random_state=42)

sample_path = "/content/drive/MyDrive/social_listening_engine/feedback/other_sample_for_keywords.csv"
other_sample.to_csv(sample_path, index=False)

print("Saved:", sample_path)
other_sample.head(5)


Saved: /content/drive/MyDrive/social_listening_engine/feedback/other_sample_for_keywords.csv


,text_clean
40624,I haven't seen any of the Sharknado movies but I'm guessing it's sort of like this.
40784,I added a video to a @user playlist FRANCIS M. - History of the Disease - March 7\u002c 20
43744,@user Pipa\u002c a tweet in support of Alonso Ezquerra. & his family. May he rest in peace
16592,On the bright side\u002c looking forward to Cambridge City away in the FA Cup tomorrow night\u002c should be a laugh! mkdons
17650,Happy Friday and don't forget about the ABC's of Yoga workshop tomorrow from 1-3 P.M. Beginners will learn...


In [ ]:

KW_ADD = {
  "complaint": [
    r"\bdoes(?:n'?t)? work\b",
    r"\bnot working\b",
    r"\bstopped working\b",
    r"\bkeeps (?:crashing|freezing|buffering)\b",
    r"\b(crash(?:ed|es)?|froze|freezing)\b",
    r"\bwon'?t (?:load|open|start|install|update)\b",
    r"\bcan'?t (?:log in|login|sign in|connect|access)\b",
    r"\bunable to (?:log in|login|sign in|connect|access)\b",
    r"\blogin (?:failed|issue|problem)\b",
    r"\b(?:error|err)\s?(?:code|message)\b",
    r"\bgetting an error\b",
    r"\bbug(?:gy)? as hell\b",
    r"\bglitch(?:y)?\b",
    r"\bso (?:annoying|frustrating)\b",
    r"\bthis is (?:annoying|frustrating)\b",
    r"\bwhat a joke\b",
    r"\bthis is ridiculous\b",
    r"\bunacceptable\b",
    r"\bdisappointed\b",
    r"\breally bummed\b",
    r"\bpissed off\b",
    r"\bwtf\b",
    r"\bbruh[, ]+this\b",
    r"\bthanks for nothing\b",
    r"\bfix (?:this|it)\b",
    r"\bplease fix (?:this|it)\b",
    r"\bstill waiting\b",
    r"\bno response\b",
    r"\bnever again\b",
    r"\bworst (?:service|support|experience)\b",
    r"\b(?:over)?charged\b",
    r"\bcharged twice\b",
    r"\bwhere is my (?:order|refund)\b",
    r"\brefund (?:me|please)\b",
    r"\bscam(?:med)?\b",
    r"\babsolute (?:trash|garbage)\b",
  ],

  "praise": [
    r"\bshoutout to\b",
    r"\bprops to\b",
    r"\bhats off\b",
    r"\bwell done\b",
    r"\bnailed it\b",
    r"\byou (?:guys|all) killed it\b",
    r"\bthat was awesome\b",
    r"\bthis is awesome\b",
    r"\bso (?:good|great|amazing)\b",
    r"\blove (?:this|it|that)\b",
    r"\babsolutely love\b",
    r"\bobsessed with\b",
    r"\bbig fan\b",
    r"\bmy favorite\b",
    r"\bcan'?t get enough\b",
    r"\bthank you so much\b",
    r"\bthanks a ton\b",
    r"\breally appreciate\b",
    r"\bappreciate (?:it|this)\b",
    r"\bmade my day\b",
    r"\bmade my week\b",
    r"\bso proud\b",
    r"\bproud supporter\b",
    r"\bso happy (?:with|about)\b",
    r"\blooks great\b",
    r"\bworks like a charm\b",
    r"\b10/10\b",
    r"\bhighly recommend\b",
    r"\bwould recommend\b",
    r"\bbest (?:ever|thing)\b",
    r"\bthe best\b",
    r"\blegend(?:ary)?\b",
    r"\bgoat(ed)?\b",
    r"\byou rock\b",
    r"\bkeep it up\b",
    r"\bthank you (?:guys|all)\b",
  ],

  "question": [
    r"\banyone know\b",
    r"\bdoes anyone know\b",
    r"\bcan someone\b",
    r"\bcan anybody\b",
    r"\bany idea\b",
    r"\bany updates?\b",
    r"\bany update\b",
    r"\bwhat'?s the update\b",
    r"\bwhat is going on\b",
    r"\bwhat happened\b",
    r"\bwhy is (?:this|it|that)\b",
    r"\bhow do i\b",
    r"\bhow to\b",
    r"\bwhere do i\b",
    r"\bwhere can i\b",
    r"\bwhere is (?:my|the)\b",
    r"\bwhen (?:is|are|will)\b",
    r"\bwhat time\b",
    r"\bwhat date\b",
    r"\bwho else\b",
    r"\bis this (?:normal|safe|legit)\b",
    r"\bshould i\b",
    r"\bdo you (?:have|know)\b",
    r"\bare you (?:guys|all)\b",
    r"\bcan i\b",
    r"\bdoes it\b",
    r"\bwill it\b",
    r"\bwhere to (?:buy|get|find)\b",
    r"\banyone else (?:having|getting)\b",
    r"\bam i the only one\b",
    r"\bhelp me understand\b",
    r"\bwhat does (?:this|that)\s+mean\b",
    r"\b(?:pls|please)\s+explain\b",
    r"\bquick question\b",
    r"\bserious question\b",
    r"\?\s*$",
  ],

  "request": [
    r"\bplease (?:help|advise|assist)\b",
    r"\bpls (?:help|advise|assist)\b",
    r"\bcan you (?:help|fix|check|confirm)\b",
    r"\bcould you (?:help|fix|check|confirm)\b",
    r"\bwould you (?:please\s+)?(?:help|fix|check)\b",
    r"\bneed (?:help|support)\b",
    r"\bi need (?:help|support)\b",
    r"\blooking for (?:help|advice|recommendations?)\b",
    r"\bcan someone (?:help|assist)\b",
    r"\banyone (?:able|free) to help\b",
    r"\bplease (?:reply|respond)\b",
    r"\bpls (?:reply|respond)\b",
    r"\bplease (?:dm|pm)\b",
    r"\bpls (?:dm|pm)\b",
    r"\bplease (?:send|share)\b",
    r"\bpls (?:send|share)\b",
    r"\bplease (?:look|check) into\b",
    r"\bcan you (?:send|share)\b",
    r"\bcould you (?:send|share)\b",
    r"\bcan you (?:recommend|suggest)\b",
    r"\bcould you (?:recommend|suggest)\b",
    r"\brecommend me\b",
    r"\bsuggest (?:me|some)\b",
    r"\bplease (?:add|include)\b",
    r"\bplease (?:remove|delete)\b",
    r"\bplease (?:cancel|pause)\b",
    r"\bplease (?:refund|replace)\b",
    r"\bpls (?:refund|replace)\b",
    r"\bfix this (?:please|pls)\b",
    r"\bhelp (?:me|us) out\b",
    r"\bdo me a favor\b",
    r"\bplease retweet\b",
    r"\bpls rt\b",
    r"\bcan you (?:RT|retweet)\b",
    r"\bfollow back\b",
    r"\bplease follow\b",
    r"\bcan you (?:follow|subscribe)\b",
  ],
}

# got the data from chatbot, i gave 300 texts, asked it do give me phrases and keywords

KW_ADD2 = {
  "conversation": [
    # personal narrative / setup
    r"\bso i was\b",
    r"\bso we were\b",
    r"\bso there i was\b",
    r"\byesterday i\b",
    r"\blast night i\b",
    r"\bthis morning i\b",
    r"\bthe other day\b",
    r"\btoday i (was|went|saw|heard|found|realized)\b",
    r"\bjust (got|saw|found|heard|realized)\b",
    r"\b(i|we) just (found out|realized|remembered)\b",
    r"\b(i|we) just (saw|heard|learned)\b",
    r"\bturns out\b",
    r"\bso apparently\b",
    r"\bplot twist\b",
    r"\bthat moment when\b",
    r"\bmeanwhile\b",
    r"\banyway[,!]\b",
    r"\blong story short\b",
    r"\btl;dr\b",
    r"\btrue story\b",
    r"\bstory time\b",

    # dialogue / retelling
    r"\b(i|we) (was|were) like\b",
    r"\b(i|we) (went|walked|drove) (to|into)\b",
    r"\bthen (he|she|they) (said|goes)\b",
    r"\bmy (friend|buddy|mom|dad|sister|brother|coworker|roommate) (said|was like)\b",
    r"\b(i|we) told (him|her|them)\b",
    r"\b(i|we) asked (him|her|them)\b",
    r"\".*\b(i|we)\b.*\"",

    # vibe / self-talk
    r"\bnot gonna lie\b",
    r"\bno joke\b",
    r"\bi swear\b",
    r"\bcan'?t stop (laughing|thinking)\b",
    r"\bwhat a (day|week)\b",
    r"\bmy (life|day) in a nutshell\b",
    r"\brandom thought\b",
    r"\bshower thought\b",
    r"\bguess what\b",
    r"\bmay or may not\b",
    r"\b(or )?may not\b",
    r"\bso i guess\b",

    # casual greetings / life updates
    r"\bhappy (birthday|bday|anniversary)\b",
    r"\bbelated happy birthday\b",
    r"\bhappy (friday|monday|tuesday|wednesday|thursday|saturday|sunday)\b",
    r"\bhow('?s| is) everyone (doing|today)\b",
    r"\bwhat are you (up to|doing) (today|tonight)\b",

    # light coordination / social chatter (not brand-action)
    r"\bwhos going to\b",
    r"\bwho's going to\b",
    r"\banyone going to\b",
    r"\bwe should (totally|totes)?\s*meet\b",
    r"\banyone else remember\b",
    r"\bremember when\b",
    r"\bcan we talk about\b",

    # common “sharing stuff”
    r"\b(i|we) added a (video|song)\b",
    r"\b(i|we) (got|added) a (video|pic|photo)\b",
    r"\bnew photo of\b",
    r"\blost & found\b",
    r"\brest in peace\b",
    r"\brip\b",
    r"\blooking forward to\b",
    r"\bcan'?t wait (to|for)\b",
  ],

  "sarcasm": [
    # classic sarcasm markers
    r"\byeah right\b",
    r"\bas if\b",
    r"\bsure jan\b",
    r"\blove that for me\b",
    r"\bmust be nice\b",
    r"\bthanks for nothing\b",
    r"\bthanks a lot\b",
    r"\bcool story bro\b",
    r"\bwhat could possibly go wrong\b",
    r"\bbecause of course\b",
    r"\bof course it (did|does|would)\b",

    # mock-positive templates
    r"\bgreat job\b",
    r"\bnice going\b",
    r"\bgood one\b",
    r"\bwell that'?s just great\b",
    r"\bkeep up the great work\b",
    r"\bjust what i needed\b",
    r"\bexactly what i wanted\b",
    r"\bhow (nice|lovely) of you\b",

    # “not” flips (high precision when paired)
    r"\bperfect\b.*\bnot\b",
    r"\bawesome\b.*\bnot\b",
    r"\bso helpful\b.*\bnot\b",
    r"\bthank you\b.*\bnot\b",
    r"\bappreciate it\b.*\bnot\b",
    r"\bworks like a charm\b.*\bnot\b",
    r"\bnailed it\b.*\bnot\b",
    r"\bgenius\b.*\bnot\b",
    r"\bbravo\b.*\bnot\b",
    r"\bwell played\b.*\bnot\b",

    # explicit sarcasm annotation
    r"\b\/s\b",
    r"\b\#sarcasm\b",
    r"\b(sarcasm|sarcastic)\b",

    # meme-y / internet sarcasm
    r"\bslow clap\b",
    r"\bclap clap\b",
    r"\bthanks captain obvious\b",
    r"\bbless your heart\b",
    r"\b10/10\b.*\brecommend\b",
    r"\bwhat a (surprise|shock)\b",
    r"\bshocking\b.*\bnot\b",

    # “yeah, sure” type (keep tight)
    r"\b(yeah|yea)\s+okay\b",
    r"\bokay[,!]*\s*sure\b",
    r"\bsuu+re\b",
    r"\brii+ght\b",
    r"\bright{2,}\b",

    # exaggerated blame / mock certainty
    r"\byeah[, ]+everything is\b.*\bfault\b",
    r"\beven if\b.*\b(it'?s|its)\s+gonna be\b.*\bfault\b",

    # sardonic consolation / backhanded framing
    r"\bon the bright side\b",
    r"\b(i guess|guess) i'll just\b",
    r"\b(i'?m|im) thrilled\b",
    r"\bcongrats\b.*\bnot\b",

    # “welcome back…don’t screw it up” style irony
    r"\bwelcome back\b.*\b(don'?t|dont)\s+(screw|mess)\s+it\s+up\b",
  ]
}


import numpy as np
import re

# --- rollback-friendly backup (do this once before overwriting) ---
events2["intent_label_prev"] = events2.get("intent_label", np.nan)
events2["intent_conf_prev"]  = events2.get("intent_conf", np.nan)
events2["needs_review_prev"] = events2.get("needs_review", np.nan)

# --- updated PRIORITY (includes new labels) ---
PRIORITY = ["sarcasm", "conversation", "request", "complaint", "question", "praise"]

# --- merge KW + KW_ADD + KW_ADD2 into one dict (dedup) ---
KW_ALL = {}
for k in set(list(KW.keys()) + list(KW_ADD.keys()) + list(KW_ADD2.keys())):
    combined = KW.get(k, []) + KW_ADD.get(k, []) + KW_ADD2.get(k, [])
    seen = set()
    KW_ALL[k] = [x for x in combined if not (x in seen or seen.add(x))]

# --- compile patterns once ---
PATTERNS = {k: [re.compile(p, re.IGNORECASE) for p in KW_ALL[k]] for k in KW_ALL}

def rule_intent(text: str):
    t = text or ""
    for cat in PRIORITY:
        hits = [p.pattern for p in PATTERNS.get(cat, []) if p.search(t)]
        if hits:
            conf = min(0.55 + 0.12 * len(hits), 0.95)
            return cat, conf, hits[:3]
    return "other", 0.40, []

tmp = events2["text_clean"].map(rule_intent)

events2["intent_rule_version"] = "rules_expanded_kw_add_kw_add2"
events2["intent_label_rule"]   = tmp.map(lambda x: x[0])
events2["intent_conf_rule"]    = tmp.map(lambda x: x[1])
events2["intent_reason_rule"]  = tmp.map(lambda x: x[2])

events2["intent_label"] = events2["intent_label_rule"]
events2["intent_conf"]  = events2["intent_conf_rule"]

events2["needs_review"] = (events2["intent_conf"] < 0.55) | (events2["intent_label"] == "other")

print("Intent distribution (expanded rules):")
print(events2["intent_label"].value_counts())

print("Other rate %:", round((events2["intent_label"] == "other").mean() * 100, 2))
print("conversation:", (events2["intent_label"] == "conversation").sum())
print("sarcasm:", (events2["intent_label"] == "sarcasm").sum())



NameError: name 'events2' is not defined

In [ ]:
out_path = "/content/drive/MyDrive/social_listening_engine/data/events_with_intent_rules_v1.parquet"
events2.to_parquet(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/social_listening_engine/data/events_with_intent_rules_v1.parquet


In [ ]:
print("events2 rule version:", events2["intent_rule_version"].dropna().unique()[:5])

print("counts:")
print(events2["intent_label"].value_counts().head(10))


events2 rule version: ['rules_expanded_kw_add_kw_add2']
counts:
intent_label
other           28163
question         6457
conversation     4426
praise           2704
request          2633
complaint        1109
sarcasm           123
Name: count, dtype: int64


##- Zero Shot Classification

Rules miss a lot though. So for the tricky cases, we let a small DistilBERT model take a shot.
Didn't run it on everything — just 5000 samples to keep things fast.

That also means rules wasn't the constraint.
we'll now make AI do zero-shot calls on this in batches.

We'll still work with v2


In [ ]:
from transformers import pipeline
import time


LABELS = ["complaint", "praise", "question", "request", "conversation", "sarcasm", "other"]

# Only used for fallback rows
CANDIDATE_LABELS = ["complaint", "praise", "question", "request", "conversation", "sarcasm"]

clf = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",
    device=-1
)

def clip(s, n=200):
    s = s or ""
    return s[:n]

other_mask = events2["intent_label"] == "other"
todo_idx = events2[other_mask].index[:5000].tolist()  # demo-cap

batch_size = 2000
n_batches = (len(todo_idx) + batch_size - 1) // batch_size
print("Fallback rows:", len(todo_idx), "batches:", n_batches)

for b in range(n_batches):
    batch_idx = todo_idx[b*batch_size:(b+1)*batch_size]
    texts = events2.loc[batch_idx, "text_clean"].tolist()
    texts = [clip(t) for t in texts]

    t0 = time.time()
    preds = clf(texts, candidate_labels=CANDIDATE_LABELS, multi_label=False)

    events2.loc[batch_idx, "intent_label_zs"] = [p["labels"][0] for p in preds]
    events2.loc[batch_idx, "intent_conf_zs"]  = [p["scores"][0] for p in preds]
    print(f"Batch {b+1}/{n_batches} done in {time.time()-t0:.1f}s")


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Fallback rows: 5000 batches: 3
Batch 1/3 done in 1160.0s
Batch 2/3 done in 1154.0s
Batch 3/3 done in 561.3s


In [ ]:
import numpy as np

# initialize from rule
events2["intent_label_final"]  = events2["intent_label"]
events2["intent_conf_final"]   = events2["intent_conf"]
events2["intent_source_final"] = "rule"

# where zero-shot exists, override
zs_mask = events2["intent_label_zs"].notna()

events2.loc[zs_mask, "intent_label_final"]  = events2.loc[zs_mask, "intent_label_zs"]
events2.loc[zs_mask, "intent_conf_final"]   = events2.loc[zs_mask, "intent_conf_zs"]
events2.loc[zs_mask, "intent_source_final"] = "zeroshot"


print("Final intent distribution:")
print(events2["intent_label_final"].value_counts())

print("\nFinal other %:",
      round((events2["intent_label_final"]=="other").mean()*100, 2))

print("\nSource breakdown:")
print(events2["intent_source_final"].value_counts())

print("\nLow confidence (<0.5) count:",
      (events2["intent_conf_final"] < 0.5).sum())


Final intent distribution:
intent_label_final
nan             40615
question         2254
conversation     1767
complaint         488
praise            301
request           160
sarcasm            30
Name: count, dtype: int64

Final other %: 0.0

Source breakdown:
intent_source_final
zeroshot    45615
Name: count, dtype: int64

Low confidence (<0.5) count: 2978


In [ ]:
print("intent_label_zs dtype:", events2["intent_label_zs"].dtype)

print("Top raw values (including weird ones):")
print(events2["intent_label_zs"].astype(str).value_counts().head(10))

print("How many are literal 'nan' strings?")
print((events2["intent_label_zs"].astype(str).str.lower() == "nan").sum())


intent_label_zs dtype: object
Top raw values (including weird ones):
intent_label_zs
nan             40615
question         2254
conversation     1767
complaint         488
praise            301
request           160
sarcasm            30
Name: count, dtype: int64
How many are literal 'nan' strings?
40615


In [ ]:
import numpy as np

# Normalize junk to real NaN
events2["intent_label_zs"] = events2["intent_label_zs"].replace(
    ["nan", "NaN", "None", "", " "], np.nan
)

# Now compute mask correctly
zs_mask = events2["intent_label_zs"].notna()

# Rebuild final columns from rules first
events2["intent_label_final"]  = events2["intent_label"]
events2["intent_conf_final"]   = events2["intent_conf"]
events2["intent_source_final"] = "rule"

# Only overwrite where zero-shot exists
events2.loc[zs_mask, "intent_label_final"]  = events2.loc[zs_mask, "intent_label_zs"]
events2.loc[zs_mask, "intent_conf_final"]   = events2.loc[zs_mask, "intent_conf_zs"]
events2.loc[zs_mask, "intent_source_final"] = "zeroshot"

print(events2["intent_label_final"].value_counts())
print("Source breakdown:\n", events2["intent_source_final"].value_counts())


intent_label_final
other           23163
question         8711
conversation     6193
praise           3005
request          2793
complaint        1597
sarcasm           153
Name: count, dtype: int64
Source breakdown:
 intent_source_final
rule        40615
zeroshot     5000
Name: count, dtype: int64


###Saving Progress

Saving the labeled data so we don't have to rerun classification every time.

In [ ]:
export_cols = [
    "event_id","text","text_clean","timestamp","timestamp_is_synthetic","source",
    "sentiment_label","sentiment_conf",
    "intent_label_final","intent_conf_final","intent_source_final",
]
events_export = events2[export_cols].copy()

out_path = "/content/drive/MyDrive/social_listening_engine/data/labeled_events.parquet"
events_export.to_parquet(out_path, index=False)
print("Saved:", out_path)

# read-back sanity
re = pd.read_parquet(out_path)
print(re.shape)
print(re.columns.tolist())
print(re["intent_source_final"].value_counts())


Saved: /content/drive/MyDrive/social_listening_engine/data/labeled_events.parquet
(45615, 11)
['event_id', 'text', 'text_clean', 'timestamp', 'timestamp_is_synthetic', 'source', 'sentiment_label', 'sentiment_conf', 'intent_label_final', 'intent_conf_final', 'intent_source_final']
intent_source_final
rule        40615
zeroshot     5000
Name: count, dtype: int64


##4- Building Search Index

Now the fun part — turning text into vectors so we can search semantically.
FAISS makes this fast even with 45k posts.

In [ ]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
!pip -q install -U sentence-transformers faiss-cpu


DATA_PATH = "/content/labeled_events.parquet"
events2 = pd.read_parquet(DATA_PATH)

print(events2.shape)
print(events2.columns.tolist())
events2.head(5)


(45615, 11)
['event_id', 'text', 'text_clean', 'timestamp', 'timestamp_is_synthetic', 'source', 'sentiment_label', 'sentiment_conf', 'intent_label_final', 'intent_conf_final', 'intent_source_final']


,event_id,text,text_clean,timestamp,timestamp_is_synthetic,source,sentiment_label,sentiment_conf,intent_label_final,intent_conf_final,intent_source_final
0,0,"""QT @user In the original draft of the 7th boo...","""QT @user In the original draft of the 7th boo...",2026-01-07 16:32:37,True,tweet_eval,pos,NaN,question,0.356723,zeroshot
1,1,"""Ben Smith / Smith (concussion) remains out of...","""Ben Smith / Smith (concussion) remains out of...",2025-12-06 05:19:00,True,tweet_eval,neutral,NaN,complaint,0.968572,zeroshot
2,2,Sorry bout the stream last night I crashed out...,Sorry bout the stream last night I crashed out...,2026-01-23 05:13:47,True,tweet_eval,neutral,NaN,conversation,0.670000,rule
3,3,Chase Headley's RBI double in the 8th inning o...,Chase Headley's RBI double in the 8th inning o...,2026-01-26 13:30:02,True,tweet_eval,neutral,NaN,conversation,0.295322,zeroshot
4,4,@user Alciato: Bee will invest 150 million in ...,@user Alciato: Bee will invest 150 million in ...,2026-02-07 18:13:44,True,tweet_eval,pos,NaN,conversation,0.398576,zeroshot


In [ ]:
print(events2.shape)
print(events2.columns.tolist())

assert "text_clean" in events2.columns, "text_clean column missing"
assert "intent_label_final" in events2.columns, "intent_label_final missing"

print("OK — required columns exist.")


(45615, 11)
['event_id', 'text', 'text_clean', 'timestamp', 'timestamp_is_synthetic', 'source', 'sentiment_label', 'sentiment_conf', 'intent_label_final', 'intent_conf_final', 'intent_source_final']
OK — required columns exist.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/social_listening_engine"
INDEX_DIR = f"{PROJECT_DIR}/index"
import os
os.makedirs(INDEX_DIR, exist_ok=True)

INDEX_PATH = f"{INDEX_DIR}/faiss.index"
META_PATH  = f"{INDEX_DIR}/faiss_meta.parquet"

print("Index path:", INDEX_PATH)
print("Meta path:", META_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Index path: /content/drive/MyDrive/social_listening_engine/index/faiss.index
Meta path: /content/drive/MyDrive/social_listening_engine/index/faiss_meta.parquet


In [ ]:

embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(embed_model_name)

TEXT_COL = "text_clean"

texts = events2[TEXT_COL].fillna("").astype(str).tolist()

emb = model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

n, d = emb.shape
print("Embeddings shape:", emb.shape)

index = faiss.IndexFlatIP(d)
index.add(emb)
print("FAISS index size:", index.ntotal)

faiss.write_index(index, INDEX_PATH)

meta_cols = [
    "event_id", "timestamp", "source",
    "sentiment_label", "intent_label_final", TEXT_COL
]
meta = events2[meta_cols].copy()
meta["row_id"] = np.arange(len(meta), dtype=np.int32)
meta.to_parquet(META_PATH, index=False)

print("Saved index + meta.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/179 [00:00<?, ?it/s]

Embeddings shape: (45615, 384)
FAISS index size: 45615
Saved index + meta.


###Quick test search -

Quick test — searching for "delivery delay refund" should bring up angry Amazon posts.
Seems to work okay.

In [ ]:
def search(query, k=10):
  q_emb = model.encode([query], normalize_embeddings = True, convert_to_numpy=True).astype('float32')
  scores, idxs = index.search(q_emb, k)
  hits = meta.iloc[idxs[0]].copy()
  hits['score'] = scores[0]
  return hits[['score', 'event_id', 'intent_label_final', 'sentiment_label', TEXT_COL]].head(k)

search('delivery delay refund not received', k=10)





,score,event_id,intent_label_final,sentiment_label,text_clean
25109,0.488865,25109,complaint,neg,"Yeah, I know it's Sunday delivery &amp; whatno..."
11676,0.475976,11676,other,neg,H&M delivery is SO shit. I ordered things and ...
26910,0.458821,26910,question,pos,Did I miss something? Just placed an order fro...
43618,0.451755,43618,complaint,neg,Finally broke down and got Amazon Prime and th...
953,0.440382,953,question,neg,Amazon Prime is a lie. It said my stuff would ...
5236,0.437411,5236,request,neg,@user paid for Amazon prime. 2nd day my produc...
12317,0.429223,12317,request,neg,@user No I want my Raspberry Pi which I ordere...
29525,0.423993,29525,request,neg,@user You may please check my pending complain...
25864,0.419574,25864,other,neg,"@user Ordered PC parts on Amazon Prime Day, on..."
36180,0.417180,36180,other,neutral,@user As our CS team have explained the last s...


In [ ]:

import pandas as pd
import numpy as np


DATA_PATH = "/content/drive/MyDrive/social_listening_engine/data/labeled_events.parquet"
events2 = pd.read_parquet(DATA_PATH)

print(events2.shape)

import faiss

INDEX_PATH = "/content/drive/MyDrive/social_listening_engine/index/faiss.index"
META_PATH  = "/content/drive/MyDrive/social_listening_engine/index/faiss_meta.parquet"

index = faiss.read_index(INDEX_PATH)

meta = pd.read_parquet(META_PATH)

print("Index size:", index.ntotal)
print("Meta shape:", meta.shape)
#If index.ntotal ≈ 45615, we're good.


from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def search(query, k=10):
    q_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idxs = index.search(q_emb, k)
    hits = meta.iloc[idxs[0]].copy()
    hits["score"] = scores[0]
    return hits[["score","event_id","intent_label_final","sentiment_label","text_clean"]]

search("late delivery refund", k=10)


(45615, 11)
Index size: 45615
Meta shape: (45615, 7)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,score,event_id,intent_label_final,sentiment_label,text_clean
11676,0.482520,11676,other,neg,H&M delivery is SO shit. I ordered things and ...
25109,0.482085,25109,complaint,neg,"Yeah, I know it's Sunday delivery &amp; whatno..."
43618,0.444963,43618,complaint,neg,Finally broke down and got Amazon Prime and th...
31915,0.444505,31915,question,neutral,@user Are you refunding booking fees due to Th...
26910,0.421829,26910,question,pos,Did I miss something? Just placed an order fro...
1982,0.420527,1982,conversation,neg,ordered my back pack off of Amazon 3 weeks ago...
37923,0.416621,37923,question,neg,My Amazon package was supposed to be delivered...
22066,0.414663,22066,complaint,neg,@user I cancelled with Amazon as they weren't ...
29525,0.412653,29525,request,neg,@user You may please check my pending complain...
953,0.412575,953,question,neg,Amazon Prime is a lie. It said my stuff would ...


##Topic discovery

assigning each post to a topic_id

We've already turned texts into vectors, now we're going to group similar vectors together, and then extract keywords for each topic making it easier to search


----

What we’re doing (simple)

- Re-embed text_clean (MiniLM) → vectors

- Cluster vectors into K topics (KMeans)

- For each topic: extract top keywords (TF-IDF) + a few example posts

- Save

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

texts = meta['text_clean'].fillna('').astype(str).tolist()


X = model.encode(
    texts, batch_size=256,
    show_progress_bar=True,
    convert_to_numpy = True,
    normalize_embeddings=True
).astype('float32')


print('Embedding Matrix: ', X.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/179 [00:00<?, ?it/s]

Embedding Matrix:  (45615, 384)


In [ ]:
#using KMeans for clustering topics

from sklearn.cluster import KMeans

K = 20

kmeans = KMeans(n_clusters=K, random_state=51, n_init='auto')
topic_id = kmeans.fit_predict(X)

meta['topic_id'] = topic_id
print(meta['topic_id'].value_counts().head())


topic_id
4     3649
5     3032
12    2765
3     2734
8     2631
Name: count, dtype: int64


In [ ]:
#keywords extraction per topic

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    stop_words='english',
    min_df=5
)

T = tfidf.fit_transform(meta['text_clean'].fillna('').astype(str))
vocab = np.array(tfidf.get_feature_names_out())

def top_kw_for_topics(tid, topn=12):
  idx = np.where(meta['topic_id'].values == tid)[0]
  if len(idx)==0:
    return[]

  mean_vec = T[idx].mean(axis=0)
  mean_vex = np.asarray(mean_vec).ravel()
  top_idx = mean_vec.argsort()[::-1][:topn]
  return vocab[top_idx].tolist()

topic_keywords = {tid: top_kw_for_topics(tid, topn=12) for tid in range(K)}
topic_keywords




{0: [['stones',
   'just noticed',
   'just money',
   'just monday',
   'just showed',
   'just seen',
   'stores',
   'storage',
   'stops',
   'stopping',
   'stop talking',
   'stoops',
   'just opened',
   'stone magazine',
   'stone cold',
   'stone brewing',
   'stone age',
   'just scored',
   'just saying',
   'just sayin',
   'just used',
   'just turned',
   'just trying',
   'just listened',
   'straight year',
   'straight win',
   'straight week',
   'straight outta',
   'story user',
   'storms',
   'storm watch',
   'storm',
   'just met',
   'just makes',
   'just little',
   'just took',
   'just reminded',
   'just remembered',
   'just remember',
   'just really',
   'just ready',
   'just ran',
   'just played',
   'just play',
   'just passed',
   'just ordered',
   'justin lenny',
   'just wanted',
   'just wanna',
   'just waiting',
   'just wait',
   'just user',
   'kandi',
   'kamran shahid',
   'kamran',
   'kalam',
   'juventus',
   'juve',
   'sticker',
  

In [ ]:
#building topics json

import json

# join labels for quick mixes
# meta has: sentiment_label, intent_label_final already

topics = []
for tid in range(K):
    sub = meta[meta["topic_id"] == tid]
    examples = sub["text_clean"].head(5).tolist()
    intent_mix = sub["intent_label_final"].value_counts(dropna=False).to_dict()
    sentiment_mix = sub["sentiment_label"].value_counts(dropna=False).to_dict()

    topics.append({
        "topic_id": int(tid),
        "size": int(len(sub)),
        "keywords": topic_keywords.get(tid, []),
        "examples": examples,
        "intent_mix": intent_mix,
        "sentiment_mix": sentiment_mix,
    })

TOPICS_PATH = "/content/drive/MyDrive/social_listening_engine/data/topics.json"
with open(TOPICS_PATH, "w") as f:
    json.dump(topics, f, indent=2)

print("Saved:", TOPICS_PATH)
print("Example topic:", topics[0])


Saved: /content/drive/MyDrive/social_listening_engine/data/topics.json
Example topic: {'topic_id': 0, 'size': 2155, 'keywords': [['stones', 'just noticed', 'just money', 'just monday', 'just showed', 'just seen', 'stores', 'storage', 'stops', 'stopping', 'stop talking', 'stoops', 'just opened', 'stone magazine', 'stone cold', 'stone brewing', 'stone age', 'just scored', 'just saying', 'just sayin', 'just used', 'just turned', 'just trying', 'just listened', 'straight year', 'straight win', 'straight week', 'straight outta', 'story user', 'storms', 'storm watch', 'storm', 'just met', 'just makes', 'just little', 'just took', 'just reminded', 'just remembered', 'just remember', 'just really', 'just ready', 'just ran', 'just played', 'just play', 'just passed', 'just ordered', 'justin lenny', 'just wanted', 'just wanna', 'just waiting', 'just wait', 'just user', 'kandi', 'kamran shahid', 'kamran', 'kalam', 'juventus', 'juve', 'sticker', 'justin biebers', 'justin beiber', 'justified', 'jus

In [ ]:
EVENTS_TOPICS_PATH = "/content/drive/MyDrive/social_listening_engine/data/events_with_topics.parquet"
meta.to_parquet(EVENTS_TOPICS_PATH, index=False)
print("Saved:", EVENTS_TOPICS_PATH)

Saved: /content/drive/MyDrive/social_listening_engine/data/events_with_topics.parquet


In [ ]:
#sanity checks
print(meta["topic_id"].value_counts().head(10))

tid = 3
print(topic_keywords[tid])
print(meta[meta["topic_id"]==tid][["intent_label_final","sentiment_label","text_clean"]].head(2))

[['homer season', 'honestly think', 'honesty', 'honey', 'honey badger', 'honored', 'honoring', 'honors', 'hoodie', 'homer', 'homs', 'homered', 'homers', 'homerun', 'homes', 'hoops', 'team 2nd', 'team win', 'teamed', 'teamfollowback', 'technical', 'home vs', 'home watch', 'home watching', 'homecoming game', 'holiday monday', 'hollande', 'holly', 'holly holm', 'hometown mar', 'teaming', 'ted', 'ted cruz', 'ted nugent', 'teddy', 'tde', 'teach', 'teaching', 'team 1st', 'homework', 'hope like', 'hopeful', 'hopeful bangladesh', 'hooked', 'hooks', 'hormones', 'horror', 'horror story', 'hospital', 'host golden', 'hoped', 'hope ll', 'hope make', 'hope milan', 'hope niall', 'hope paul', 'tc', 'tcot', 'tcu', 'hostages', 'td pass', 'tay', 'taylor allderdice', 'taylor kitsch', 'taylor swift', 'tb', 'hope does', 'hope doesn', 'hope don', 'honestly don', 'home sunday', 'td run', 'targets', 'tarheels', 'task', 'task force', 'taste', 'tasting', 'tate', 'tattoo', 'hm', 'hit 2nd', 'hit run', 'hitler', 'h

What we just did-

We:

- Turned each post into a semantic vector (embedding)

- Grouped similar vectors using KMeans into ~20 clusters

- Treated each cluster as a topic

- Extracted top keywords per topic using TF-IDF

- Assigned every post a topic_id

- Built topics.json with:
  - keywords

  - example posts

  - intent mix

  - sentiment mix



So now the engine can answer:

“What are people talking about?”
and
“Within that topic, are they complaining, requesting, praising, etc.?”

##Future Upgrade (B): UMAP + HDBSCAN (BERTopic-style)

What would change:

- UMAP (dimensionality reduction)
Compress embeddings (e.g., 384 → ~5–15 dims) while preserving structure.
Makes clusters cleaner and separable.

- HDBSCAN (density-based clustering)
Finds clusters of variable sizes automatically.
Can mark true outliers as “noise” (no forced topic).

- c-TF-IDF (class-based TF-IDF)
Extracts keywords per cluster more robustly than plain TF-IDF.

Result:

- No need to pick K beforehand

- Better separation of niche topics

- Real “misc/outlier” bucket

- More stable keywords


Tradeoff:

- Heavier deps

- Slower on free Colab

- Slightly more tuning

As this is a  demo, KMeans is the stable choice; other option is the “accuracy upgrade”.

In [ ]:
# new(final) starting point

import json, os
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

PROJECT_DIR = "/content/drive/MyDrive/social_listening_engine"
INDEX_PATH = f"{PROJECT_DIR}/index/faiss.index"
META_PATH  = f"{PROJECT_DIR}/index/faiss_meta.parquet"
TOPICS_PATH = f"{PROJECT_DIR}/data/topics.json"

index = faiss.read_index(INDEX_PATH)
meta = pd.read_parquet(META_PATH)

topics = json.load(open(TOPICS_PATH, "r"))
topics_by_id = {t["topic_id"]: t for t in topics}

for t in topics:
    k = t.get("keywords", [])
    if isinstance(k, list) and len(k) == 1 and isinstance(k[0], list):
        t["keywords"] = k[0]

topics_by_id = {t["topic_id"]: t for t in topics}
print("Normalized topic keywords.")

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Loaded:", index.ntotal, meta.shape, "topics:", len(topics))


Normalized topic keywords.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded: 45615 (45615, 7) topics: 20


In [ ]:
import numpy as np

def retrieve(query, k=50, intent=None, sentiment=None, topic_id=None):
    q = (query or "").strip()
    if not q:
        return meta.head(0).copy()

    q_emb = model.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idxs = index.search(q_emb, k)

    hits = meta.iloc[idxs[0]].copy()
    hits["score"] = scores[0]

    # filters
    if intent:
        hits = hits[hits["intent_label_final"] == intent]
    if sentiment:
        hits = hits[hits["sentiment_label"] == sentiment]
    if topic_id is not None:
        hits = hits[hits["topic_id"] == int(topic_id)] if "topic_id" in hits.columns else hits

    return hits.sort_values("score", ascending=False)


In [ ]:
TOPIC_META_PATH = f"{PROJECT_DIR}/data/events_with_topics.parquet"

if os.path.exists(TOPIC_META_PATH):
    meta_topics = pd.read_parquet(TOPIC_META_PATH)[["row_id","topic_id"]]
    meta = meta.merge(meta_topics, on="row_id", how="left")
    print("Merged topic_id into meta. Null topic_id:", meta["topic_id"].isna().sum())
else:
    print("events_with_topics.parquet not found. trending/topics will be limited.")


Merged topic_id into meta. Null topic_id: 0


##5- Trend Analysis

Checking which topics are spiking recently vs their normal baseline.
This is how you spot emerging conversations.

In [ ]:
# creating topic trend function

def trend_topics(days_recent=3, days_baseline=14, top_n=10):
  df = meta.copy()
  df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
  df = df.dropna(subset=['timestamp', 'topic_id'])
  df['day'] = df['timestamp'].dt.floor('D')

  last_day = df['day'].max()
  recent_start = last_day - pd.Timedelta(days=days_recent-1)
  base_start = last_day - pd.Timedelta(days=days_recent+days_baseline)
  base_end = last_day - pd.Timedelta(days = days_recent)

  daily = df.groupby(['day', 'topic_id']).size().reset_index(name='n')

  recent = daily[daily["day"] >= recent_start].groupby("topic_id")["n"].sum()
  base   = daily[(daily["day"] >= base_start) & (daily["day"] <= base_end)].groupby("topic_id")["n"].sum()

  trend = pd.DataFrame({"recent": recent, "baseline": base}).fillna(0)
  trend["trend_score"] = (trend["recent"] + 1) / (trend["baseline"] + 1)  # stable ratio
  trend = trend.sort_values("trend_score", ascending=False).head(top_n).reset_index()

  # attach keywords
  trend["keywords"] = trend["topic_id"].map(lambda tid: topics_by_id.get(int(tid), {}).get("keywords", [])[:8])
  return trend

#check
trend_topics(top_n=10)

,topic_id,recent,baseline,trend_score,keywords
0,13,70,286,0.247387,"[day, th, birthday, hot dog, national, cream d..."
1,2,99,417,0.239234,"[game, sunday, th, night, brady, tom brady, fo..."
2,18,84,355,0.238764,"[kasich, john kasich, obama, trump, carly, fio..."
3,16,95,405,0.236453,"[th, tomorrow, st, just, gucci, google, ibm, new]"
4,14,82,353,0.234463,"[tomorrow, movie, th, jurassic, just, sharknad..."
5,9,83,362,0.231405,"[just, th, like, niall, tomorrow, justin, st, ..."
6,5,110,499,0.222000,"[album, kendrick, friday, th, just, song, kend..."
7,19,73,340,0.217009,"[concert, tomorrow, tickets, th, going, jason ..."
8,7,77,373,0.208556,"[th, sox, david, mets, white sox, red sox, st,..."
9,6,28,140,0.205674,"[christians, marriage, amendment, gay marriage..."


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def themes_from_hits(hits_df, top_terms=12):
    col = "text_for_topics" if "text_for_topics" in hits_df.columns else "text_clean"
    texts = hits_df[col].fillna("").astype(str).tolist()
    if not texts:
        return []

    v = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1,2),
        max_features=8000
    )
    M = v.fit_transform(texts)
    vocab = np.array(v.get_feature_names_out())
    scores = np.asarray(M.mean(axis=0)).ravel()
    top_idx = scores.argsort()[::-1][:top_terms]

    return vocab[top_idx].tolist()


def trend_from_hits(hits_df, freq="D"):
    if "timestamp" not in hits_df.columns or hits_df.empty:
        return pd.DataFrame()

    df = hits_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    df["date"] = df["timestamp"].dt.floor(freq)

    trend = (
        df.groupby(["date", "sentiment_label"])
          .size()
          .reset_index(name="count")
          .sort_values("date")
    )

    return trend

In [ ]:
#this was redone after editing the code below

from collections import Counter
import pandas as pd

def build_report(query, top_k=20, intent=None, sentiment=None, topic_id=None, k_trend=500):
    # 1) Pull a larger set for trend (but still bounded)
    k_big = max(k_trend, top_k, 50)
    hits_big = retrieve(
        query,
        k=k_big,
        intent=intent,
        sentiment=sentiment,
        topic_id=topic_id
    )

    # 2) UI subset (what we show as evidence)
    hits = hits_big.head(top_k).copy()

    # ensure topic-clean text exists for themes
    if "text_for_topics" not in hits.columns and "text_for_topics" in meta.columns:
        hits = hits.merge(meta[["row_id","text_for_topics"]], on="row_id", how="left")

    # breakdowns computed on UI hits (what user sees)
    intent_counts = hits["intent_label_final"].value_counts().to_dict()
    sent_counts   = hits["sentiment_label"].value_counts().to_dict()
    topic_counts  = hits["topic_id"].value_counts().to_dict() if "topic_id" in hits.columns else {}

    # query-aware topics (within UI hits)
    top_topics = []
    if "topic_id" in hits.columns:
        for tid, cnt in hits["topic_id"].value_counts().head(5).items():
            t = topics_by_id.get(int(tid), {})
            kw = t.get("keywords", [])
            if isinstance(kw, list) and len(kw) == 1 and isinstance(kw[0], list):
                kw = kw[0]
            top_topics.append({
                "topic_id": int(tid),
                "count": int(cnt),
                "keywords": kw[:10]
            })

    # themes from UI hits (aligned)
    themes = themes_from_hits(hits, top_terms=12)

    # bullets
    bullets = []
    if intent_counts:
        bullets.append(f"Most retrieved posts look like **{max(intent_counts, key=intent_counts.get)}**.")
    if sent_counts:
        bullets.append(f"Sentiment is mostly **{max(sent_counts, key=sent_counts.get)}**.")
    if themes:
        bullets.append(f"Top themes: {', '.join(themes[:8])}")
    bullets.append(f"Trend computed from top {len(hits_big)} retrieved matches (evidence shows top {len(hits)}).")

    # evidence JSON-safe
    ev = hits[[
        "score","event_id","timestamp",
        "intent_label_final","sentiment_label","text_clean"
    ]].copy()
    ev["timestamp"] = ev["timestamp"].astype(str)
    evidence = ev.to_dict(orient="records")

    # trends computed on BIG hits (JSON-safe)
    trend_df = trend_from_hits(hits_big)
    if not trend_df.empty and "date" in trend_df.columns:
        trend_df["date"] = trend_df["date"].astype(str)

    report = {
        "query": query,
        "filters": {"intent": intent, "sentiment": sentiment, "topic_id": topic_id},
        "bullets": bullets,
        "themes": themes,
        "breakdowns": {
            "intent": intent_counts,
            "sentiment": sent_counts,
            "topics": topic_counts
        },
        "top_topics": top_topics,
        "trend": trend_df.to_dict(orient="records"),
        "evidence": evidence
    }
    return report

In [ ]:
report = build_report("delivery delay refund", top_k=20)
report["bullets"]
report["themes"]

['prime',
 'user',
 'amazon',
 'amazon prime',
 'delivery',
 'day',
 'sunday',
 'package',
 'shipping',
 'supposed',
 'ordered',
 'tomorrow']

In [ ]:
#saving the report

report = build_report("delivery delay refund", top_k=20)
REPORT_PATH = f"{PROJECT_DIR}/data/insights_report.json"

import json
with open(REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print("Saved:", REPORT_PATH)
report["bullets"], len(report["evidence"])


Saved: /content/drive/MyDrive/social_listening_engine/data/insights_report.json


(['Most retrieved posts look like **question**.',
  'Sentiment is mostly **neg**.',
  'Top themes: prime, user, amazon, amazon prime, delivery, day, sunday, package',
  'Trend computed from top 500 retrieved matches (evidence shows top 20).'],
 20)

In [ ]:
tid = report["top_topics"][0]["topic_id"]
print(meta[meta["topic_id"]==tid].sample(5)["text_clean"].tolist())

print(meta.shape)
print("topic_id nulls:", meta["topic_id"].isna().sum())

["<Tech News> Apple's iPhone event: Join us Wednesday (live blog) The Lam Research Theater at the Yerba Buena Center for the Arts in San Fr", 'See news through the eyes of real people &amp; be your own reporter with Fresco 2.0 for iOS, coming September 8th!', 'Moto G (2nd Generation) Just Rs.8099 pay with SBI card Black White', '@user You may please check my pending complaint with Amazon for which I write many emails and called customer care.', "Grrrrrrr need a new profile pic but can't find the right one And this stupid iPad can't take photos cause I have the 1st one grrrrrrr"]
(45615, 8)
topic_id nulls: 0


In [ ]:
# better keyword cleaner - v2 bascially

import re

def clean_for_topics(s: str) -> str:
  s = (s or '').lower()
  s = re.sub(r"@\w+", " ", s)            # mentions
  s = re.sub(r"http\S+", " ", s)         # urls
  s = re.sub(r"#(\w+)", r" \1 ", s)       # keep hashtag word, drop '#'
  s = re.sub(r"[^a-z\s]", " ", s)         # letters + spaces only
  s = re.sub(r"\s+", " ", s).strip()
  return s

meta['text_for_topics'] = meta['text_clean'].map(clean_for_topics)
meta['text_for_topics'].head(10)

,text_for_topics
0,qt in the original draft of the th book remus ...
1,ben smith smith concussion remains out of the ...
2,sorry bout the stream last night i crashed out...
3,chase headley s rbi double in the th inning of...
4,alciato bee will invest million in january ano...
5,lit my mum kerry the louboutins i wonder how m...
6,soul train oct halloween special ft t dot fine...
7,so disappointed in wwe summerslam i want to se...
8,this is the last sunday w o football nfl is ba...
9,cena aj sitting in a tree k i s s i n g st goe...


In [ ]:
# rebuilding
from sklearn.feature_extraction.text import TfidfVectorizer

# aggregating docs per topic into one 'super doc'

topic_docs = (
    meta.dropna(subset=["topic_id"])
        .groupby("topic_id")["text_for_topics"]
        .apply(lambda x: " ".join(x.tolist()))
)

vec = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    max_features=30000,
    min_df=2
)
X = vec.fit_transform(topic_docs.values)
vocab = np.array(vec.get_feature_names_out())

def top_terms_row(row_vec, topn=12):
    arr = row_vec.toarray().ravel()
    top_idx = arr.argsort()[::-1][:topn]
    return vocab[top_idx].tolist()

topic_keywords_v2 = {}
for i, tid in enumerate(topic_docs.index.tolist()):
    topic_keywords_v2[int(tid)] = top_terms_row(X[i], topn=12)

# quick peek
list(topic_keywords_v2.items())[:2]

[(0,
  ['gaga',
   'jenner',
   'nicki',
   'lady gaga',
   'caitlyn',
   'caitlyn jenner',
   'th',
   'just',
   'like',
   'tomorrow',
   'lady',
   'st']),
 (1,
  ['tomorrow',
   'night',
   'watch',
   'tonight',
   'friday',
   'big',
   'sunday',
   'just',
   'big brother',
   'brother',
   'saturday',
   'pm'])]

In [ ]:
# updating .json file in drive

for t in topics:
  tid = int(t['topic_id'])
  if tid in topic_keywords_v2:
    t['keywords'] = topic_keywords_v2[tid]

TOPICS_PATH = f"{PROJECT_DIR}/data/topics.json"
with open(TOPICS_PATH, "w") as f:
    json.dump(topics, f, indent=2)

print("Updated topics.json:", TOPICS_PATH)

# rebuild lookup
topics_by_id = {t["topic_id"]: t for t in topics}


#sanity check
tid = 10
print("Keywords:", topics_by_id[tid]["keywords"])
meta[meta["topic_id"]==tid]["text_clean"].sample(10, random_state=42).tolist()

Updated topics.json: /content/drive/MyDrive/social_listening_engine/data/topics.json
Keywords: ['apple', 'moto', 'amazon', 'gen', 'moto rd', 'rd gen', 'iphone', 'ipad', 'galaxy', 'apple watch', 'amazon prime', 'ios']


['Looks like the iPad Mini will be released 11/2. And Apple ordered 10 million for Q4 alone. Holy cow.',
 '@user I accidentally bought the 5th episode of Game of Thrones on PS3 instead of PS4. Can you put me in touch with someone to fix?',
 "@user it's fun and it may make your time in the big brother final stretch a little less painful. It's on demand and the lifetime app",
 'Finally broke down and got Amazon Prime and the delivery on my 1st purchase as a member is late. @user I was expecting better.',
 'YouTube Gaming Officially Launches On Web, Android, iOS On August 26: YouTube is finally going to r... webseries',
 "@user I don't know what is going on with your phone but what's going on tomorrow for Hype park?",
 "AppleEvent is about to start! Watch Apple's live stream here:",
 '@user I would really love to get a PS Vita. My sisters 16th is coming soon and she would be overjoyed if she gets one. CKRComp',
 'Samsung Unpacked 13 August: Where to watch live stream of Galaxy Note 5 laun

In [ ]:
pd.DataFrame(report["evidence"])[
    ["score","intent_label_final","sentiment_label","text_clean"]
].head(5)

,score,intent_label_final,sentiment_label,text_clean
0,0.519335,complaint,neg,"Yeah, I know it's Sunday delivery &amp; whatno..."
1,0.478379,other,neg,H&M delivery is SO shit. I ordered things and ...
2,0.463494,question,pos,Did I miss something? Just placed an order fro...
3,0.461999,request,neg,@user No I want my Raspberry Pi which I ordere...
4,0.460029,complaint,neg,Finally broke down and got Amazon Prime and th...


In [ ]:
# saving all the important stuff again, just to be sure

import os

print("FAISS index exists:", os.path.exists(INDEX_PATH))
print("Meta exists:", os.path.exists(META_PATH))
print("Topics exists:", os.path.exists(TOPICS_PATH))

TOPIC_META_PATH = f"{PROJECT_DIR}/data/events_with_topics.parquet"
print("Topic merge file exists:", os.path.exists(TOPIC_META_PATH))

FAISS index exists: True
Meta exists: True
Topics exists: True
Topic merge file exists: True


##6- Gradio Dashboard

Wrapping everything in a simple UI so it's not just a notebook.
Lets you search, filter by intent/sentiment, and see trends.

In [ ]:
!pip install gradio

import gradio as gr
import json

def gradio_search(query, intent_filter, sentiment_filter):
  intent_filter = None if intent_filter == 'Any' else intent_filter
  sentiment_filter = None if sentiment_filter == 'Any' else sentiment_filter

  report = build_report(
      query,
      top_k=20,
      intent = intent_filter,
      sentiment = sentiment_filter
  )

  bullets_text = '\n'.join(report['bullets'])
  themes_text = ', '.join(report.get('themes', []))

  evidence_df = pd.DataFrame(report['evidence'])[
      ['score', 'intent_label_final', 'sentiment_label', 'text_clean']
  ]

  return bullets_text, themes_text, evidence_df, json.dumps(report, indent=2)

In [ ]:
import pandas as pd
import numpy as np

def compute_trending_topics(meta, topics_by_id, days_recent=7, days_base=30, min_base=15, top_n=15):
    df = meta.copy()
    df = df.dropna(subset=["topic_id", "timestamp"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    end = df["timestamp"].max()
    start_recent = end - pd.Timedelta(days=days_recent)
    start_base = start_recent - pd.Timedelta(days=days_base)

    recent = df[(df["timestamp"] >= start_recent) & (df["timestamp"] <= end)]
    base   = df[(df["timestamp"] >= start_base) & (df["timestamp"] < start_recent)]

    recent_counts = recent["topic_id"].value_counts()
    base_counts   = base["topic_id"].value_counts()

    out = []
    for tid, rc in recent_counts.items():
        bc = int(base_counts.get(tid, 0))
        if bc < min_base:
            continue

        # rate-normalized lift (so windows can differ)
        recent_rate = rc / max(days_recent, 1)
        base_rate   = bc / max(days_base, 1)
        lift = recent_rate / (base_rate + 1e-9)

        kw = topics_by_id.get(int(tid), {}).get("keywords", [])
        if isinstance(kw, list) and len(kw) == 1 and isinstance(kw[0], list):
            kw = kw[0]
        kw = ", ".join([str(x) for x in kw[:8]])

        out.append({
            "topic_id": int(tid),
            "lift": float(lift),
            "recent_count": int(rc),
            "base_count": int(bc),
            "keywords": kw
        })

    res = pd.DataFrame(out)
    if res.empty:
        return res, {"end": end, "start_recent": start_recent, "start_base": start_base}

    res = res.sort_values(["lift", "recent_count"], ascending=False).head(top_n).reset_index(drop=True)
    return res, {"end": end, "start_recent": start_recent, "start_base": start_base}


def topic_examples(meta, topic_id, n=8, seed=42):
    df = meta[meta["topic_id"] == int(topic_id)].copy()
    if df.empty:
        return []
    df = df.sample(min(n, len(df)), random_state=seed)
    return df["text_clean"].fillna("").astype(str).tolist()

In [ ]:
trend_df, windows = compute_trending_topics(meta, topics_by_id, days_recent=7, days_base=30)
trend_df.head(5), windows

(   topic_id      lift  recent_count  base_count  \
 0        18  1.253602           203         694   
 1        12  1.217362           244         859   
 2         9  1.123272           195         744   
 3        13  1.107716           145         561   
 4         0  1.077448           179         712   
 
                                             keywords  
 0  kasich, john kasich, obama, trump, carly, fior...  
 1  real madrid, madrid, milan, tomorrow, th, arse...  
 2  just, th, like, niall, tomorrow, justin, st, time  
 3  day, th, birthday, hot dog, national, cream da...  
 4  gaga, jenner, nicki, lady gaga, caitlyn, caitl...  ,
 {'end': Timestamp('2026-02-17 23:58:23'),
  'start_recent': Timestamp('2026-02-10 23:58:23'),
  'start_base': Timestamp('2026-01-11 23:58:23')})

In [ ]:
import gradio as gr
import pandas as pd
import json
import traceback
import matplotlib.pyplot as plt
import os, re, tempfile
from datetime import datetime

def _safe_slug(s: str, max_len=60):
    s = (s or "query").lower()
    s = re.sub(r"[^a-z0-9]+", "-", s).strip("-")
    return s[:max_len] or "query"

def plot_trend(trend_df: pd.DataFrame):
    if trend_df is None or trend_df.empty:
        fig = plt.figure()
        plt.title("No trend data")
        return fig

    df = trend_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    fig = plt.figure()
    for s, g in df.groupby("sentiment_label"):
        g = g.sort_values("date")
        plt.plot(g["date"], g["count"], label=str(s))

    plt.legend()
    plt.title("Query Trend (by sentiment)")
    plt.xlabel("date")
    plt.ylabel("count")
    plt.xticks(rotation=30)
    plt.tight_layout()
    return fig


# --- SEARCH TAB logic (stores report in state, downloads on button) ---
def gradio_search(query, intent_filter, sentiment_filter, top_k):
    try:
        intent_filter = None if intent_filter == "Any" else intent_filter
        sentiment_filter = None if sentiment_filter == "Any" else sentiment_filter

        report = build_report(
            query,
            top_k=int(top_k),
            intent=intent_filter,
            sentiment=sentiment_filter
        )

        bullets_text = "\n".join(report.get("bullets", []))
        themes_text = ", ".join(report.get("themes", []))

        evidence_df = pd.DataFrame(report.get("evidence", []))
        if not evidence_df.empty:
            keep = [c for c in ["score","intent_label_final","sentiment_label","text_clean"] if c in evidence_df.columns]
            evidence_df = evidence_df[keep]

        trend_df = pd.DataFrame(report.get("trend", []))
        trend_plot = plot_trend(trend_df)

        report_json = json.dumps(report, indent=2)

        return bullets_text, themes_text, evidence_df, trend_plot, trend_df, report_json, report

    except Exception:
        err = traceback.format_exc()
        empty_plot = plot_trend(pd.DataFrame())
        return "ERROR", "", pd.DataFrame(), empty_plot, pd.DataFrame(), err, None


def download_report(report):
    if report is None:
        return None

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = _safe_slug(report.get("query", "query"))
    filename = f"insights_{ts}_{slug}.json"
    tmp_path = os.path.join(tempfile.gettempdir(), filename)

    with open(tmp_path, "w") as f:
        json.dump(report, f, indent=2)

    return tmp_path


# --- TRENDING TOPICS TAB logic ---
def gradio_trending(days_recent, days_base, min_base, top_n):
    try:
        trend_df, windows = compute_trending_topics(
            meta,
            topics_by_id,
            days_recent=int(days_recent),
            days_base=int(days_base),
            min_base=int(min_base),
            top_n=int(top_n)
        )

        win_text = (
            f"End: {windows['end']}\n"
            f"Recent window: {windows['start_recent']} → {windows['end']}  (last {days_recent} days)\n"
            f"Baseline window: {windows['start_base']} → {windows['start_recent']}  (prev {days_base} days)"
        )

        return trend_df, win_text
    except Exception:
        return pd.DataFrame(), traceback.format_exc()


def gradio_topic_examples(topic_id):
    try:
        if topic_id is None or str(topic_id).strip() == "":
            return ""
        ex = topic_examples(meta, int(topic_id), n=8)
        return "\n\n---\n\n".join(ex)
    except Exception:
        return traceback.format_exc()


with gr.Blocks() as demo:
    gr.Markdown("# Social Listening Insight Engine (Demo)")

    with gr.Tabs():
        # ---------------- SEARCH TAB ----------------
        with gr.Tab("Search"):
            state_report = gr.State(None)

            with gr.Row():
                query = gr.Textbox(label="Query", placeholder="e.g. delivery delay refund", scale=3)
                top_k = gr.Slider(5, 50, value=20, step=5, label="Top K (evidence shown)", scale=1)

            with gr.Row():
                intent_filter = gr.Dropdown(
                    ["Any","complaint","question","request","praise","conversation","sarcasm","other"],
                    value="Any",
                    label="Intent Filter"
                )
                sentiment_filter = gr.Dropdown(
                    ["Any","pos","neg","neutral"],
                    value="Any",
                    label="Sentiment Filter"
                )

            with gr.Row():
                search_btn = gr.Button("Run")
                download_btn = gr.Button("Download JSON")

            bullets_out = gr.Textbox(label="Summary", lines=5)
            themes_out = gr.Textbox(label="Themes")
            evidence_out = gr.Dataframe(label="Evidence", wrap=True)

            trend_plot_out = gr.Plot(label="Trend Chart")
            trend_table_out = gr.Dataframe(label="Trend Table", wrap=True)

            json_out = gr.Textbox(label="Raw JSON / Errors", lines=10)
            download_out = gr.File(label="Download File")

            search_btn.click(
                gradio_search,
                inputs=[query, intent_filter, sentiment_filter, top_k],
                outputs=[bullets_out, themes_out, evidence_out, trend_plot_out, trend_table_out, json_out, state_report]
            )

            download_btn.click(
                download_report,
                inputs=[state_report],
                outputs=[download_out]
            )

        # ---------------- TRENDING TOPICS TAB ----------------
        with gr.Tab("Trending Topics"):
            gr.Markdown("Shows topics rising in the recent window vs baseline (global, not query-only).")

            with gr.Row():
                days_recent = gr.Slider(3, 21, value=7, step=1, label="Recent window (days)")
                days_base   = gr.Slider(7, 90, value=30, step=1, label="Baseline window (days)")
                min_base    = gr.Slider(0, 200, value=15, step=1, label="Min baseline count (filter)")
                top_n       = gr.Slider(5, 50, value=15, step=5, label="Top N topics")

            trend_btn = gr.Button("Compute Trending Topics")

            windows_out = gr.Textbox(label="Windows used", lines=3)
            trending_table = gr.Dataframe(label="Trending topics", wrap=True)

            gr.Markdown("Pick a topic_id from the table and paste it below to see example posts.")
            topic_id_in = gr.Number(label="topic_id")
            examples_btn = gr.Button("Show examples")
            examples_out = gr.Textbox(label="Example posts", lines=14)

            trend_btn.click(
                gradio_trending,
                inputs=[days_recent, days_base, min_base, top_n],
                outputs=[trending_table, windows_out]
            )

            examples_btn.click(
                gradio_topic_examples,
                inputs=[topic_id_in],
                outputs=[examples_out]
            )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bb4c697df5ca3f7e21.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://bb4c697df5ca3f7e21.gradio.live
